# Lab 13: Multivariate OLS — What "Controlling For" Actually Does
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 80 min Core + 40 min AI + 35 min Extension + take-home Challenge

---

**Learning Objectives:**
- Show that a coefficient's meaning changes when a second variable enters
- Derive omitted variable bias and predict its sign before running the regression
- Demonstrate Frisch-Waugh-Lovell: "controlling for" is literally partialling out
- Distinguish a control that removes bias from one that creates it
- Check a specification table written by AI against coefficients you computed yourself

**Dataset:** Simulated data with a known truth: wages, education and unobserved ability; a collider; two nearly identical regressors; and union pay scales. The Extension uses California Housing.

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions. Part 4 runs one guided step, then has three one-line blanks, in Step 7. The Extension is optional, and the Challenge is a take-home. The AI-Assisted Expansion at the end is the one place you use AI to write code, and you check what it writes against your own numbers.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 4, Step 7:** three one-line blanks (`____`): the regression with the interaction, and the return to a year of experience for a non-union worker and for a union member.
> - The answers to the interpretation questions after each part, in text cells. Part 2's Question 2 also asks for a few lines of code, in a new code cell.
> - **The AI Expansion:** version 2 of the given prompt, revised once, after you find what version 1 left the AI to guess. Then four short answers: your revision, the check against your own coefficients from Steps 2 and 5, the key lines in your own words, and when AI is worth using for this.
>
> **You run and read:** everything else. Setup, Parts 1-3, the first cell of Part 4, and the check cell in the AI Expansion are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The AI's function is pasted, not typed, and the code itself is not graded; your answers about it are.
>
> **The Extension** (optional) and **the Challenge** (take-home) have no starter code: you write them in new cells, from the task. The **README** is text only: the prompt at the end is for documentation, not code.

## Setup

Every random number in this lab comes from one generator, `rng`, and each draw moves it on. So
your numbers match the expected output only if you run the cells once, from the top, in order. If
you run a cell twice, use **Runtime → Restart session** and run from the top again.

In [ ]:
# GUIDED — run as-is
# Step 1: imports and the random generator
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor   # used in Step 6

rng = np.random.default_rng(seed=42)    # every simulated draw below comes from rng, in order

print("Setup complete.")

## Part 1: Omitted Variable Bias, With the Truth Known (GUIDED — 15 min)

Simulation is the only setting where you can check a bias formula against the
bias, so we start there.

Step 2 simulates wages that depend on education and on ability, then runs two regressions. The
**short** one leaves ability out, as you would have to if ability were unobserved; the **long** one
puts it in. `sm.OLS` and `sm.add_constant` work as in Lab 10's Step 5: in `.params`, index 0 is the
intercept and index 1 is the coefficient on the first regressor, here `educ`. `.bse` holds the
standard errors, as `.params` holds the coefficients.

In [ ]:
# GUIDED — run as-is
# Step 2: wages depend on education AND on ability; then the short and the long regression
n = 5_000                                             # 5_000 is 5000; the _ only makes it easier to read
ability = rng.normal(0, 1, n)
educ = 12 + 1.5 * ability + rng.normal(0, 1.5, n)     # abler people study longer
wage = 20_000 + 2_500 * educ + 8_000 * ability + rng.normal(0, 5_000, n)

TRUE_RETURN = 2_500                                   # the 2_500 * educ in the line above

short = sm.OLS(wage, sm.add_constant(educ)).fit()                                # ability left out
long_ = sm.OLS(wage, sm.add_constant(np.column_stack([educ, ability]))).fit()   # ability in; the _ is part of the name

# <34 pads the text to 34 characters; >12,.0f right-aligns a number in 12, with commas and no
# decimals; >10.2f right-aligns one in 10, with 2 decimals
print(f"{'specification':<34}{'educ coef':>12}{'se':>10}{'truth':>10}")
print("-" * 68)
print(f"{'wage ~ educ  (ability omitted)':<34}{short.params[1]:>12,.0f}{short.bse[1]:>10.2f}{TRUE_RETURN:>10,}")
print(f"{'wage ~ educ + ability':<34}{long_.params[1]:>12,.0f}{long_.bse[1]:>10.2f}{TRUE_RETURN:>10,}")

**Expected output:** with ability left out, the coefficient on education is 5,126, about twice the
true 2,500. With ability in, it is 2,546, close to the truth. The standard errors are 50.98 and
46.54.

Step 3 predicts that gap without the long regression. The omitted-variable-bias formula says the
short coefficient is off by **gamma × delta**: gamma is the effect of the omitted variable (ability)
on the outcome, and delta is the slope from regressing the omitted variable on the included one
(ability on education).

In [ ]:
# GUIDED — run as-is
# Step 3: the bias formula, gamma x delta, against the actual bias
delta = sm.OLS(ability, sm.add_constant(educ)).fit().params[1]   # the slope of ability on educ
gamma = 8_000         # ability's true effect on wage, the 8_000 * ability in Step 2

print(f"gamma (ability -> wage)      = {gamma:>10,.0f}")
print(f"delta (educ -> ability)      = {delta:>10.4f}")
print(f"predicted bias  gamma*delta  = {gamma * delta:>10,.0f}")
print(f"actual bias                  = {short.params[1] - TRUE_RETURN:>10,.0f}")

**Expected output:** gamma is 8,000 and delta is 0.3286, so the formula predicts a bias of 2,629.
The actual bias, 5,126 − 2,500, is 2,626.

### Interpretation Questions

1. The bias is positive. Work through the formula and say which of the two terms would have to flip for the short regression to *understate* the return to education.
2. In real data, ability is unobserved. What is left of this exercise when you cannot run the long regression?
3. Chapter 10 called the omitted variable a confounder. Draw the DAG for this simulation and mark the backdoor path.

*Your answers here:*

## Part 2: Frisch-Waugh-Lovell (GUIDED — 20 min)

"Controlling for X" is not just a phrase: it is a calculation you can do, and this part does it
by hand.

Step 4 does it in three lines. A **residual** is what a regression leaves unexplained: the actual
value minus the fitted one, and `.resid` holds one per row. The first line strips out of education
the part that ability explains, the second strips the same out of wages, and the third regresses
what is left of wages on what is left of education.

In [ ]:
# GUIDED — run as-is
# Step 4: recover the long regression's coefficient on educ from three short regressions
educ_resid = sm.OLS(educ, sm.add_constant(ability)).fit().resid   # education that ability does not explain
wage_resid = sm.OLS(wage, sm.add_constant(ability)).fit().resid   # wages that ability does not explain
fwl = sm.OLS(wage_resid, sm.add_constant(educ_resid)).fit()       # leftover on leftover

print(f"multivariate coefficient on educ : {long_.params[1]:>12,.4f}")
print(f"FWL two-step coefficient         : {fwl.params[1]:>12,.4f}")

**Expected output:** the same number twice, 2,546.4985: Step 2's long regression and the FWL
two-step agree exactly, not approximately. "Holding ability fixed" means: strip the part of
education that ability explains, strip the same from wages, and relate what is left.

### Interpretation Questions

1. State in one sentence what `educ_resid` contains, in words a non-economist would follow.
2. Step 4 residualised both sides. Predict what you would get if you residualised only the outcome, then check.
3. Chapter 24 defines Double ML as "extending FWL". Given what you just did, what part of this two-step would a machine learning model be doing there?

*Your answers here:*

## Part 3: The Control That Creates Bias (GUIDED — 25 min)

Chapter 10 warned about this and Chapter 13 shows it inside a regression table. Step 5 is the
collider from Lab 10's Step 6, drawn again: D and Y are independent, and both cause C.

In [ ]:
# GUIDED — run as-is
# Step 5: a collider control. D and Y are independent by construction
n = 10_000
d = rng.normal(0, 1, n)
y = rng.normal(0, 1, n)                        # no arrow d -> y
c = 1.2 * d + 1.2 * y + rng.normal(0, 1, n)    # both cause c

no_ctrl = sm.OLS(y, sm.add_constant(d)).fit()
with_ctrl = sm.OLS(y, sm.add_constant(np.column_stack([d, c]))).fit()

print(f"{'specification':<24}{'estimate':>10}{'se':>8}{'p':>10}{'truth':>8}")
print("-" * 60)
print(f"{'y ~ d':<24}{no_ctrl.params[1]:>10.3f}{no_ctrl.bse[1]:>8.4f}{no_ctrl.pvalues[1]:>10.3f}{0.0:>8.1f}")
print(f"{'y ~ d + c':<24}{with_ctrl.params[1]:>10.3f}{with_ctrl.bse[1]:>8.4f}{with_ctrl.pvalues[1]:>10.3f}{0.0:>8.1f}")

**Expected output:** without the control, 0.001 (standard error 0.0101) with p = 0.958, on a truth
of zero. With c as a control, −0.596 (standard error 0.0082), and a p-value that prints as 0.000,
meaning below 0.0005. The uncontrolled regression was right; adding a control invented an effect.

Step 6 turns to a different worry about controls: two regressors so alike that the regression can
hardly tell them apart (**multicollinearity**). The **variance inflation factor** (VIF) of a
regressor is 1 / (1 − R²), where R² comes from regressing it on the other regressors. A VIF of 1
means the others explain none of it; the larger the VIF, the more of it they already explain.
Collinearity multiplies a regressor's standard error by √VIF, compared with the case where it is
uncorrelated with the others.

In [ ]:
# GUIDED — run as-is
# Step 6: multicollinearity. x2 is x1 plus a little noise
x1 = rng.normal(0, 1, 2_000)
x2 = x1 + rng.normal(0, 0.08, 2_000)            # nearly the same variable
yv = 3 * x1 + 2 * x2 + rng.normal(0, 1, 2_000)  # true effects 3 and 2

Xc = pd.DataFrame({"x1": x1, "x2": x2})
fit = sm.OLS(yv, sm.add_constant(Xc)).fit()
Xv = sm.add_constant(Xc).values      # the same regressors as a plain array, which variance_inflation_factor takes

print(f"{'term':>6}{'coef':>10}{'std err':>10}{'p':>8}{'VIF':>10}")
print("-" * 46)
# enumerate gives each name with its position i. Fitted on a DataFrame, .params is labelled
# by name, so .iloc[i] picks an entry by position
for i, name in enumerate(["const", "x1", "x2"]):
    print(f"{name:>6}{fit.params.iloc[i]:>10.3f}{fit.bse.iloc[i]:>10.3f}"
          f"{fit.pvalues.iloc[i]:>8.3f}{variance_inflation_factor(Xv, i):>10.1f}")

**Expected output:** three rows. `x1` and `x2` each have a VIF of 150.4 and a standard error of
0.276, and their coefficients are 3.262 and 1.748, against true effects of 3 and 2. Ignore the
`const` row's VIF of 1.0: VIF describes a regressor, and the constant is not one.

In [ ]:
# GUIDED — run as-is
# Step 6, continued: what collinearity damages, and what it does not
infl = np.sqrt(variance_inflation_factor(Xv, 1))   # how many times larger x1's standard error is

# .2e prints in scientific notation: 1.50e-03 is 0.0015
print(f"Joint F-test p-value: {fit.f_pvalue:.2e}")
print(f"Standard errors are {infl:.0f}x what they would be if x1 and x2 were uncorrelated")
print(f"x1 + x2 coefficients: {fit.params['x1'] + fit.params['x2']:.3f}   (truth 3 + 2 = 5)")

**Expected output:** the joint F-test p-value prints as 0.00e+00, so small that it rounds to zero.
The standard errors are 12 times what they would be without the collinearity (√150.4 ≈ 12.3). The
two coefficients add up to 5.009, close to the true 5.

The split between the two coefficients is imprecise: it moves a lot from sample to sample. But the
joint test is overwhelming, and the sum is close to the truth. Collinearity inflates standard
errors; it does not bias the estimates.

### Interpretation Questions

1. Both Part 1 and Part 3 show a coefficient moving when a control is added. What distinguishes the two cases, and where does that information come from?
2. A colleague drops `x2` because its VIF is high. What have they done to the interpretation of the coefficient on `x1`?
3. When is a high VIF genuinely a problem you should act on?

*Your answers here:*

## Part 4: Interactions (YOUR TASK — 20 min)

So far every coefficient has been one number for everybody. Often the effect
depends on who you are.

Step 7 simulates pay in which a year of experience is worth 900 to a non-union worker, and 400 less
to a union member, whose pay scale is flatter. The column `exper_union` is experience times the
union dummy: it equals experience for a union member and 0 for everyone else. A regression that
includes it, beside `exper` and `union`, lets the slope on experience differ between the groups.

In [ ]:
# GUIDED — run as-is
# Step 7: union pay, where the return to experience is lower for union members
n = 4_000
union = rng.binomial(1, 0.3, n)          # 1 for a union member (30% of workers), 0 otherwise
exper = rng.uniform(0, 30, n)            # years of experience, anywhere from 0 to 30
pay = (30_000 + 900 * exper + 6_000 * union
       - 400 * exper * union + rng.normal(0, 4_000, n))

frame = pd.DataFrame({"exper": exper, "union": union,
                      "exper_union": exper * union, "pay": pay})
frame.head()

**Expected output:** the first five rows of `frame`. Where `union` is 1, `exper_union` equals
`exper`; where it is 0, `exper_union` is 0.

Now fit the regression and read the two returns off it. `sm.OLS` takes a DataFrame of regressors as
well as an array, and `frame[["a", "b"]]` picks two columns as a DataFrame. Fitted that way,
`.params` is labelled by column name, as in Lab 10's Step 10, where the trainee effect was
`params["treat"]`. `summary()` is the full regression report, and `.tables[1]` is its table of
coefficients.

In [ ]:
# YOUR TASK — fill in the three blanks (____)
# Step 7, continued: fit the interaction, then read the two returns off it
# pay on exper, union and exper_union, with a constant
fit_int = ____
print(fit_int.summary().tables[1])

# union is 0 for a non-union worker and 1 for a union member
return_nonunion = ____
return_union = ____

print(f"Return to a year of experience, non-union: {return_nonunion:>8,.0f}")
print(f"Return to a year of experience, union:     {return_union:>8,.0f}")

**Expected output:** a table of four rows (`const`, `exper`, `union` and `exper_union`), then the
two returns. They should land near the truth built into Step 7: 900 a year for a non-union worker,
and 900 − 400 = 500 for a union member. *NameError: name '____' is not defined* means a blank is
still empty.

### Interpretation Questions

1. The coefficient on `union` alone is not "the union effect". Say what it actually is, and at what value of experience it applies.
2. Your two returns differ. Is that difference statistically distinguishable from zero? Which number in the table tells you?
3. If you had fitted `pay ~ exper + union` without the interaction, what would the experience coefficient have represented?

*Your answers here:*

---

## Extension (Optional — 35 min): Choosing What Goes In

You now have two rules pulling in opposite directions: omitted variables bias
you, and bad controls bias you. There is no procedure that resolves this from
the data.

**Build a specification table.** Using California Housing, estimate the association between
`AveRooms` and house value under at least five different control sets, from no controls to
everything available. There is no starter code: write it in the cell below and in new cells under
it. Load the data as Lab 12 did:

```python
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True).frame.rename(columns={"MedHouseVal": "value"})
```

A shape that works, if you want one: put the control sets in a list of lists, loop over it, fit
`sm.OLS` each time, and append a dict of `{"controls": ..., "coef": ..., "se": ..., "r2": ...}` to
a results list. `pd.DataFrame(results)` at the end gives you the table. That is the same
loop-and-collect pattern as Lab 7's dictionary note. Five control sets:

```python
specs = [
    [],                                        # no controls
    ["MedInc"],
    ["MedInc", "HouseAge"],
    ["MedInc", "HouseAge", "Population"],
    [col for col in housing.columns if col not in ("AveRooms", "value")],   # everything
]
```

If you have done the AI-Assisted Expansion below, and its `spec_table` matched your numbers, you
may use it here instead of writing the loop.

**Then answer:** the coefficient will move, possibly a lot. Write four sentences on what a reader
should conclude from a coefficient that is unstable across specifications, and what they should
conclude from one that is perfectly stable.

In [ ]:
# YOUR TASK (optional) — Extension: write your code here, and in new cells below

---

## Challenge (Take-Home): Sign the Bias Before You Run It

The most useful skill in this chapter is predicting a bias from an argument
rather than discovering it from output. There is no starter code: write your answers in the cell
below and in new cells under it.

**Predict the sign of the bias, then check.** For each scenario, write down the predicted sign of
the bias before writing any code, then build a simulation, like Step 2's, that checks whether you
were right.

- (a) Estimating the effect of class size on test scores, omitting household income.
- (b) Estimating the effect of a training programme on wages, omitting motivation.
- (c) Estimating the returns to a degree, controlling for occupation. (Careful: this one is not an
  omitted variable problem.)

Report which of your three predictions were wrong, and why. Being wrong once here is normal; what
matters is that you notice.

In [ ]:
# YOUR TASK — take-home Challenge: write your code here, and in new cells below

---

## AI-Assisted Expansion: a specification table written by AI, checked by you

The lecture's procedure for choosing controls ends with "report the alternatives anyway": one row
per set of controls, side by side, so a reader sees how the estimate moves. Steps 2 and 5 each did
that by hand for two specifications. The loop, the fits and the table are repetitive code that AI
writes well, and this is a fair test of AI, because you already know the numbers the table must
contain: your coefficients from Steps 2 and 5.

**What to do**
1. Copy **prompt version 1** below into Claude ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT.
2. Read what comes back before you run anything. The prompt leaves at least one
   important thing unsaid, so the AI had to guess. Find one guess that is wrong,
   or one that you would rather decide yourself. Pasting the version 1 function into
   the empty code cell and running the check is one way to find it.
3. Write **prompt version 2** in the cell below, fixing that one thing, and ask
   again. Paste the function from version 2 into the empty code cell and run the
   check. If every number matches, find one choice in the function your prompt never
   asked for, and make version 2 state it.
4. Answer the questions at the end. They are what is graded, not the code.

**Reading the check.** It puts your arrays into two DataFrames, `wages` (Step 2's `wage`, `educ`
and `ability`) and `collider` (Step 5's `y`, `d` and `c`), and calls the AI's function on each,
first with no controls and then with the one control. It prints the AI's coefficient and standard
error beside the ones in your own fits: `short` and `long_` from Step 2, `no_ctrl` and `with_ctrl`
from Step 5, whose coefficients and standard errors Steps 2 and 5 printed. Step 2 printed them
rounded; the check prints four decimals, and a `same?` column after each pair says `yes`, or `NO`
and how far apart the two numbers are, in percent.

Your fits and the AI's use the same arrays in the same session, so a correct function agrees in
every printed digit. Any difference, even in the last digit, means it computes or reports
something differently.

The check calls `spec_table(wages, "wage", "educ", [[], ["ability"]])`, so keep the name and the
order of the four arguments. Paste only the function: if the AI also wrote example data, leave it
out, because it can replace the lab's arrays. A function that fails gets Python's error message in
place of its numbers.

In [ ]:
# PROMPT, version 1 — copy it exactly as it is
#
# I am an economics student using statsmodels in Python. Write a function
# spec_table(df, outcome, treatment, control_sets) that makes a specification
# table like the ones in applied economics papers: the same OLS regression,
# with a constant, run once for each set of controls. df is a pandas DataFrame;
# outcome and treatment are column names; control_sets is a list of lists of
# column names, with [] for no controls. Return a pandas DataFrame with one row
# per control set, in the order given, and four columns: controls (the names
# joined by ", "), coef and se (the treatment's coefficient and its standard
# error), and n (the number of rows used). I will run it on simulated wages,
# with columns wage, educ and ability. Comment each line.

# PROMPT, version 2 — your revision. Keep everything from version 1 that was
# fine, and add or change the part the AI had to guess.
#
#

In [ ]:
# Paste the AI's spec_table function here: version 1 first, then version 2. Run the check below after each.

In [ ]:
# CHECK — runs the AI's spec_table on your data from Steps 2 and 5, beside your own fits
# A fit keeps the data it ran on: .model.endog is the outcome, and .model.exog the regressors,
# with the constant in column 0. If the pasted code drew data of its own, these no longer match
lab_data = [(wage, long_.model.endog), (educ, long_.model.exog[:, 1]), (ability, long_.model.exog[:, 2]),
            (y, with_ctrl.model.endog), (d, with_ctrl.model.exog[:, 1]), (c, with_ctrl.model.exog[:, 2])]
data_unchanged = all(np.array_equal(now, then) for now, then in lab_data)   # True if every pair is equal

def same(mine, ai):
    """'yes' if two numbers agree, otherwise NO and how far apart they are, in percent."""
    # rtol=1e-9: equal to about 9 significant digits, so only rounding inside the computer passes
    if np.isclose(mine, ai, rtol=1e-9, atol=0):
        return "yes"
    # :+.2g shows the sign and 2 significant digits, so a tiny gap prints like +9.4e-06
    return f"NO {100 * (ai - mine) / abs(mine):+.2g}%"

# globals() holds every name defined so far in this notebook
if not data_unchanged:
    print("The pasted code changed the lab's data (wage, educ, ability, y, d or c). "
          "Paste only the spec_table function, then rerun the notebook from the top.")
elif "spec_table" not in globals():
    print("Paste the AI's spec_table function in the cell above first.")
else:
    wages = pd.DataFrame({"wage": wage, "educ": educ, "ability": ability})
    collider = pd.DataFrame({"y": y, "d": d, "c": c})
    # Each entry: the data, outcome, treatment and control sets, then a label and your fit for each row
    cases = [(wages, "wage", "educ", [[], ["ability"]],
              [("wage ~ educ", short), ("wage ~ educ + ability", long_)]),
             (collider, "y", "d", [[], ["c"]],
              [("y ~ d", no_ctrl), ("y ~ d + c", with_ctrl)])]

    print(f"{'':<23}{'coef, yours':>13}{'coef, AI':>13}{'same?':>14}"
          f"{'se, yours':>11}{'se, AI':>11}{'same?':>14}")
    for data, outcome, treatment, control_sets, rows in cases:
        # try runs the AI's function and reads its two columns; if anything raises an error,
        # the lines under except run instead of stopping the cell
        try:
            table = spec_table(data, outcome, treatment, control_sets)
            ai_coef = [float(v) for v in table["coef"]]     # the AI's coefficients, as plain numbers
            ai_se = [float(v) for v in table["se"]]
        except Exception as err:
            # type(err).__name__ is the kind of error, such as KeyError
            print(f"{outcome} ~ {treatment}: the AI's table could not be read. {type(err).__name__}: {err}")
            print("  spec_table must give back a DataFrame with number columns coef and se.")
            continue
        if len(ai_coef) != len(rows):
            print(f"{outcome} ~ {treatment}: the AI's table has {len(ai_coef)} rows for {len(rows)} control sets.")
            continue
        # zip walks the three lists side by side; (label, mine) unpacks each row's label and your fit
        for (label, mine), coef, se in zip(rows, ai_coef, ai_se):
            print(f"{label:<23}{mine.params[1]:>13,.4f}{coef:>13,.4f}{same(mine.params[1], coef):>14}"
                  f"{mine.bse[1]:>11,.4f}{se:>11,.4f}{same(mine.bse[1], se):>14}")

    print("\nIf a pair differs, find out why before you trust the table.")

### Document the interaction

This is what is graded for the expansion.

1. **Your revision.** What did you change between version 1 and version 2, and
   what did the AI get wrong or have to guess that made you change it? If every
   number matched the first time, name the choice the AI made on its own, and say
   how version 2 now states it.
2. **The check.** Which pairs matched, and which differed at first? For a pair
   that differed, what was the cause, and which number should the table report?
   How do you know?
3. **In your own words.** Pick the line of the AI's function that fits each
   regression (the one with `.fit`), and the line that takes the treatment's
   coefficient and standard error out of the results. Explain in 2–3 sentences
   what each does, and point to the lines of Step 2 that do the same jobs: the one that fits
   `short`, and the one that prints its coefficient (`.params`) and standard error (`.bse`).
4. **Would you use AI for this again?** One sentence: when is it worth asking AI
   to write table code like this, and when is it faster to write it yourself?

*Your answers here:*

1.
2.
3.
4.

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Simulated wages with an unobserved ability term. Leaving ability out
  gave a return to education of [YOUR VALUE] against a true 2,500;
  putting it in gave [YOUR VALUE]
* Checked the omitted-variable-bias formula: it predicted a bias of
  [YOUR VALUE], and the actual bias was [YOUR VALUE]
* Showed Frisch-Waugh-Lovell: regressing the part of wages that ability
  does not explain on the part of education that ability does not
  explain gave the same coefficient as the full regression
* Showed a collider control turning a true effect of zero into an
  estimate of [YOUR VALUE], with a p-value that printed as 0.000
* Separated what multicollinearity damages (standard errors, about
  [YOUR VALUE] times larger) from what it does not (the estimates)
* Fitted an interaction: a year of experience was worth [YOUR VALUE]
  to non-union workers and [YOUR VALUE] to union members
* Asked an AI to write a specification-table function, revised my
  prompt once, and checked its coefficients against my own regressions

**Please write a README.md entry including:**
1. Project Title: Multivariate OLS — What "Controlling For" Actually Does
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'comprehensive' or
'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab13-multivariate-ols`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab13-multivariate-ols`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab13-multivariate-ols`, branch `main`,
   commit message `Lab 13 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.